<a href="https://colab.research.google.com/github/arildbn/bban4040/blob/main/martra-notebooks/2-2_2-3_chromadb-server-client.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<div>
    <h1 style="text-align: center;">Large Language Models Projects</h1>
    <h3>Apply and Implement Strategies for Large Language Models</h3>
    <h2>2.2 / 2.3 ChromaDB Server Mode &amp; Client</h2>
    <p>by <b>Pere Martra</b></p>
</div>

_________
This notebook merges **2.2 ChromaDB Server Mode** and **2.3 ChromaDB Client** into a single notebook, so the server and the client run in the **same runtime**.

The data-loading code is the same as in `2-1_vector-databases-llms.ipynb`. We then start Chroma in server mode in the background and connect to it with `chromadb.HttpClient`. Because the server and client share one runtime, this works both locally and in a single Google Colab runtime.
__________

In [ ]:
# === Colab dependency guard (bban4040) ===
# Installing chromadb upgrades transitive packages (notably opentelemetry-*)
# past the exact versions Colab's preinstalled google packages pin (google-adk,
# the otlp / gcp-logging exporters), which prints noisy "pip's dependency
# resolver ... is incompatible" errors. We pin those families to the versions
# already installed so the install below leaves them untouched. PIP_CONSTRAINT
# is honored by every %pip call in this kernel. Harmless off Colab (nothing
# matches / nothing to pin).
import os, tempfile
from importlib import metadata

_keep = []
for _dist in metadata.distributions():
    _name = (_dist.metadata.get("Name") or "").strip()
    if not _name:
        continue
    if _name.lower() == "requests" or _name.lower().startswith("opentelemetry"):
        _keep.append(f"{_name}=={_dist.version}")

if _keep:
    _con = os.path.join(tempfile.gettempdir(), "bban4040_pip_constraints.txt")
    with open(_con, "w") as _f:
        _f.write("\n".join(sorted(set(_keep))) + "\n")
    os.environ["PIP_CONSTRAINT"] = _con
    print(f"Pinned {len(_keep)} package(s) to avoid Colab pip resolver conflicts.")

In [ ]:
%pip install chromadb

In [ ]:
# === portable-setup (bban4040) ===
# Secrets resolve from Colab "Secrets" (userdata) on Colab, or environment
# variables / a local .env file when running locally. Nothing is hardcoded.
import os
try:
    from dotenv import load_dotenv
    load_dotenv()
except Exception:
    pass


def get_secret(name, default=None):
    try:
        from google.colab import userdata
        value = userdata.get(name)
        if value:
            return value.strip()
    except Exception:
        pass
    value = os.environ.get(name, default)
    return value.strip() if isinstance(value, str) else value


_key = get_secret("OPENAI_API_KEY")
if _key:
    os.environ["OPENAI_API_KEY"] = _key


In [ ]:
import numpy as np
import pandas as pd

In [ ]:
# The dataset is mirrored on GitHub, so no Kaggle account or token is needed.
# We download it once to a local CSV (identical behaviour on Colab and locally).
import os
import urllib.request

DATA_URL = "https://raw.githubusercontent.com/kotartemiy/topic-labeled-news-dataset/master/labeled_newscatcher_dataset.csv"
csv_path = "labeled_newscatcher_dataset.csv"
if not os.path.exists(csv_path):
    urllib.request.urlretrieve(DATA_URL, csv_path)

news = pd.read_csv(csv_path, sep=';')
MAX_NEWS = 1000
DOCUMENT = "title"
TOPIC = "topic"

In [ ]:
#Because it is just a example we select a small portion of News.
subset_news = news.head(MAX_NEWS)

In [ ]:
import chromadb

In [ ]:
chroma_client = chromadb.PersistentClient(path="./chromadb")

In [ ]:
collection = chroma_client.get_or_create_collection(name="local_news_collection")

In [ ]:
collection.add(
    documents=subset_news[DOCUMENT].tolist(),
    metadatas=[{TOPIC: topic} for topic in subset_news[TOPIC].tolist()],
    ids=[f"id{x}" for x in range(MAX_NEWS)],
)

In [ ]:
results = collection.query(query_texts=["laptop"], n_results=10 )

print(results)

## Run Chroma in server mode

We start the Chroma server in the background so the client cells below can connect to it from the same runtime over `127.0.0.1`. We start it on an explicit IPv4 address (rather than `localhost`) so the server and client always agree on the address family — otherwise, on IPv6-preferring machines the server can bind to `::1` while the health check waits forever on `127.0.0.1`.

In [ ]:
import shutil
import socket
import subprocess
import sys
import tempfile
import time

# Bind to 127.0.0.1 (not "localhost") so the server and the client always agree
# on the address family. On IPv6-preferring machines "localhost" resolves to
# ::1, and Chroma's server would bind there while an IPv4 health check waited
# forever on 127.0.0.1 — pinning both sides to 127.0.0.1 avoids that hang.
CHROMA_HOST = "127.0.0.1"
CHROMA_PORT = 8000


def _port_is_open(host, port):
    with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as sock:
        sock.settimeout(1)
        return sock.connect_ex((host, port)) == 0


# `server` stays None if Chroma is already running, so the cleanup cell at the
# bottom can tell whether this notebook owns the process.
server = None

if _port_is_open(CHROMA_HOST, CHROMA_PORT):
    print(f"Port {CHROMA_PORT} already in use — assuming Chroma is already running.")
else:
    # Prefer the `chroma` console script; fall back to invoking the same entry
    # point (chromadb.cli.cli:app) via `python -c` if the script isn't on PATH.
    chroma_bin = shutil.which("chroma")
    cmd = ([chroma_bin] if chroma_bin
           else [sys.executable, "-c", "from chromadb.cli.cli import app; app()"])
    cmd += ["run", "--path", "./chromadb", "--host", CHROMA_HOST, "--port", str(CHROMA_PORT)]

    # Send the server's output to a log file rather than subprocess.PIPE.
    # An unread PIPE fills its small (~64 KB) OS buffer and deadlocks the
    # server mid-startup, and calling .read() on a still-running server's
    # PIPE blocks forever (EOF only arrives when the process exits). A file
    # never blocks and lets us show the logs if start-up fails.
    log = tempfile.NamedTemporaryFile(
        prefix="chroma-server-", suffix=".log", delete=False
    )
    server = subprocess.Popen(cmd, stdout=log, stderr=subprocess.STDOUT)

    def _server_log():
        log.flush()
        with open(log.name) as fh:
            return fh.read()

    try:
        for _ in range(60):
            if _port_is_open(CHROMA_HOST, CHROMA_PORT):
                break
            rc = server.poll()
            if rc is not None:
                raise RuntimeError(
                    f"Chroma exited with code {rc} before binding the port.\n"
                    f"--- server log ---\n{_server_log()}"
                )
            time.sleep(1)
        else:
            raise RuntimeError(
                f"Chroma server did not start on {CHROMA_HOST}:{CHROMA_PORT} "
                f"within 60s.\n--- server log ---\n{_server_log()}"
            )
    except Exception:
        server.terminate()  # don't leave a half-started server holding the port
        raise

print(f"Chroma server is ready on {CHROMA_HOST}:{CHROMA_PORT}")

## Connect with the Chroma client

Now we connect to the running Chroma server with `chromadb.HttpClient` and query the collection created above — the workflow that previously lived in `2-3_chromadb-client.ipynb`.

In [ ]:
client = chromadb.HttpClient(host=CHROMA_HOST, port=CHROMA_PORT)

In [ ]:
collection_local = client.get_collection(name="local_news_collection")
results = collection_local.query(query_texts=["laptop"], n_results=10)

In [ ]:
print(results)

In [ ]:
# Stop the background Chroma server when you are done.
# `server` is None if the server was already running before this notebook
# started it, in which case there is nothing for us to stop.
if server is not None:
    server.terminate()
    server.wait(timeout=10)
    print("Chroma server stopped.")
else:
    print("No server was started by this notebook; nothing to stop.")